## Train Advanced Model

### imports & setup

In [7]:
import pandas as pd
import numpy as np
import torch
import tqdm
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


### data preparation

In [9]:
df = pd.read_parquet('data.parquet')

In [10]:
np.random.seed(67)
all_games = df['game_id'].unique()
train_games = np.random.choice(all_games, size=int(0.8 * len(all_games)), replace=False)
train_df = df[df['game_id'].isin(train_games)]
test_df = df[~df['game_id'].isin(train_games)]

print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Train set size: 40485
Test set size: 10706


In [ ]:
def window_data(df, N=10):
    all_features = []

    for _, game_df in df.groupby('game_id'):
        game_df = game_df.sort_values('timestamp')
        if len(game_df) <= N: continue

        for i in range(N, len(game_df)):
            prices = game_df['price_close'].iloc[i-N : i+1].values
            price_trend = " -> ".join([f"{p:.2f}" for p in prices])
            current_vol = game_df['volume'].iloc[i]
            market_context = f"MARKET DATA: Prices [{price_trend}] | Volume: {current_vol:.0f} | "
            reddit_text = " </s> ".join(game_df['all_text_clean'].iloc[i-N : i+1])
            combined_text = market_context + reddit_text

            rows = {
                'text_feature': combined_text,
                'label': game_df['label'].iloc[i]
            }
            all_features.append(rows)

    return pd.DataFrame(all_features)


In [12]:
train_win = window_data(train_df, N=10)
test_win = window_data(test_df, N=10)

label_map = {'DOWN': 0, 'FLAT': 1, 'UP': 2}
train_win['label'] = train_win['label'].map(label_map)
test_win['label'] = test_win['label'].map(label_map)
print(f"Windowed Train: {len(train_win)}")
print(f"Windowed Test:  {len(test_win)}")

Windowed Train: 38775
Windowed Test:  10276


### running model

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
def tokenize_function(examples):
    return tokenizer(
        examples["text_feature"],
        padding="max_length",
        truncation=True,
        max_length=512
    )


train_dataset = Dataset.from_pandas(train_win[['text_feature', 'label']])
test_dataset = Dataset.from_pandas(test_win[['text_feature', 'label']])

train_encoded = train_dataset.map(tokenize_function, batched=True)
test_encoded = test_dataset.map(tokenize_function, batched=True)

train_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average="macro")}

model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)

training_args = TrainingArguments(
    output_dir="./roberta_results",
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    report_to="none",
    dataloader_pin_memory=True,
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model.to(device),
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=test_encoded,
    compute_metrics=compute_metrics,
)

trainer.train()